IT法務コンサルタントAI：学習データセット クレンジング
このノートブックでは、以下の3つのステップを通じて、高精度な学習データセット train_data_v7_final.jsonl を作成します。

1.クレンジング: v6（500件）から専門外の相談を除去（401件）

2.拒絶データの注入: 専門外の相談に対し「回答できない」と断るデータを追加（70件）

3.最終統合: データを結合し、学習効率を高めるためにランダムにシャッフル

In [ ]:
import json
import random

# ==========================================
# STEP 1: 元データのクレンジング
# ==========================================
# 500件の元データから、ITエンジニアが誤って相談しがちな
# 「専門外（特許、労務、税務等）」のキーワードを含むデータを除去します。

INPUT_FILE = "train_data_v6.jsonl"

# 試行錯誤の結果たどり着いた、401件に絞り込むためのキーワードリスト
REJECT_KEYWORDS = [
    "特許", "商標", "意匠", "銀行法", "資金決済法", "割賦販売法", 
    "労働基準法", "労基法", "給与", "採用", "雇用", "税務", "確定申告", 
    "医師法", "薬機法", "医療機器", "建築基準法", "独占禁止法", 
    "会社法", "株主総会", "取締役", "不倫", "埋蔵金", "カレー", "スクラム",
    "離婚", "公務員", "相続","実用新案", "不正競争防止法", "営業秘密", "ドメイン紛争",
    "仮想通貨", "暗号資産", "ステーブルコイン", "eKYC", "犯収法", "マネロン",
    "退職金", "就業規則", "不当解雇", "残業", "社会保険", "労災","診断", "治療", 
    "薬事", "健康診断", "バイタル","裁判", "訴訟", "損害賠償", "告訴", "警察",
    "名誉毀損", "誹謗中傷"
]

clean_data = []

try:
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            # キーワードが含まれていないものだけを「クリーンデータ」として採用
            if not any(kw in data.get("input", "") for kw in REJECT_KEYWORDS):
                clean_data.append(data)
    print(f"✅ Step 1 完了: {len(clean_data)} 件のクリーンデータを抽出しました。")
except FileNotFoundError:
    print(f"❌ エラー: {INPUT_FILE} が見倒りません。アップロードを確認してください。")

STEP 2: 拒絶パターンの作成
モデルに「自分の守備範囲」を教え込むため、あえて専門外の質問を投げ、それに対して「専門外であるため回答できない」と理由付きで拒絶するデータを追加します。これにより、AIが「知らないことを知ったかぶりして答える」リスクを抑えます。

In [ ]:
# ==========================================
# STEP 2: 拒絶データの追加（70件）
# ==========================================

# Geminiが生成した拒絶用トレーニングデータをここに統合します
# (文字数制限の関係上、ここでは代表的なサンプルを格納しています。
# 実際にはここに全70件のリストを挿入して実行します)

rejection_data = [
    {"instruction": "IT法務コンサルタントとして、守備範囲外の相談には適切に回答を拒絶してください。", "input": "自社のAIモデルを学習させるために、競合他社のウェブサイトから商品価格や在庫情報を1秒間に100回ペースでスクレイピングし続けたい。サーバーに負荷がかかるかもしれないけど、公開されている情報だから法的リスクはないよね？", "output": "リスクレベル：判定不可\n該当法：刑法（電子計算機損壊等業務妨害罪）、民法（不法行為）\n理由：本モデルはITプロダクトの仕様上の法的リスク（個情法、著作権法等）を対象としており、サーバーへの高負荷による業務妨害の成立可否や、不法行為に基づく損害賠償リスクの具体的判断は守備範囲外です。\n修正案：スクレイピングの頻度や利用規約の遵守について、IT紛争に強い弁護士に相談し、相手方サーバーへの影響を最小限に抑える設計を検討してください。"},
    {"instruction": "IT法務コンサルタントとして、守備範囲外の相談には適切に回答を拒絶してください。", "input": "アプリ内で「友達を1人招待するごとに、現金1000円をユーザーの銀行口座に直接振り込む」というキャンペーンをやりたい。これって何か特別な免許がいるの？それともただの広告宣伝費でOK？", "output": "リスクレベル：判定不可\n該当法：資金決済法（資金移動業）\n理由：本モデルはアプリのUI/UXや一般的法務を対象としており、ユーザーへの現金送金業務が「資金移動業」の登録を要するかどうかのライセンス判断は守備範囲外です。\n修正案：送金代行サービスの利用を検討するか、フィンテック専門の弁護士にご相談ください。"},
    # ※本セル実行時に合計70件のリストになるように調整
]

# クリーンデータと拒絶データをマージ
final_dataset = clean_data + rejection_data
print(f"✅ Step 2 完了: 拒絶データを追加し、合計 {len(final_dataset)} 件になりました。")

STEP 3: 最終統合とランダム・シャッフル
最後に、回答データと拒絶データが交互に現れるようにランダムに並び替えます。これは、学習プロセスにおいて特定のパターンが固まって出現することで生じる「バイアス（偏り）」を防ぎ、モデルの汎用性能を高めるために不可欠な工程です。

In [ ]:
# ==========================================
# STEP 3: シャッフル ＆ 最終保存
# ==========================================

OUTPUT_FINAL = "train_data_v7_final.jsonl"

# 再現性のためにシード値を固定してシャッフル
random.seed(42)
random.shuffle(final_dataset)

# JSONL形式で書き出し
with open(OUTPUT_FINAL, 'w', encoding='utf-8') as f:
    for item in final_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"🚀 すべての工程が完了しました！")
print(f"最終ファイル: {OUTPUT_FINAL} (合計 {len(final_dataset)} 件)")
print("左メニューの「ファイル」アイコンからダウンロードして、Fine-tuningに使用してください。")